[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_06/notebook_6_3_preprocessing.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# 6.3 Medical Image Preprocessing

## Learning Objectives
1. Understand modality-specific preprocessing (CT vs MRI vs X-ray)
2. Implement intensity normalization techniques
3. Apply CT windowing for specific anatomical structures
4. Perform MRI bias field correction
5. Use CLAHE (Contrast Limited Adaptive Histogram Equalization)
6. Assess image quality

## Clinical Context

**Journey 3 (Jamal - Lung Nodules)**: CT scans require proper windowing to visualize lung tissue vs mediastinum vs bone

**Journey 4 (Elena - Brain Tumor)**: MRI scans need intensity normalization and bias field correction for consistent segmentation

**Journey 5 (Priya - Melanoma)**: Dermatology images need illumination correction and color normalization

**Why Preprocessing Matters:**
- **Intensity standardization**: Different scanners, protocols → different intensity ranges
- **Quality enhancement**: Improve SNR, reduce artifacts
- **Model performance**: Proper preprocessing critical for CNN generalization
- **Clinical interpretation**: Windowing highlights relevant anatomy

---

## Setup

In [1]:
# ── Setup (JupyterLite) ─────────────────────────────────────────────────────
import sys
if sys.platform == "emscripten":          # JupyterLite / Pyodide
    import micropip
    await micropip.install(["seaborn"])

In [2]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import ndimage
from scipy.ndimage import gaussian_filter
from skimage import exposure, filters, restoration
from skimage.util import random_noise
from skimage.morphology import disk
from skimage.filters import rank
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ Libraries imported successfully!")

✓ Libraries imported successfully!


## Part 1: Intensity Normalization

### Why Normalize?

Medical images from different scanners/protocols have different intensity ranges:
- **CT**: Hounsfield Units (HU) are standardized (-1000 to +3000)
- **MRI**: Arbitrary units (vary by scanner, sequence, coil)
- **X-ray**: Pixel values depend on acquisition parameters

**CNNs expect consistent input ranges** (typically [0, 1] or [-1, 1]).

In [3]:
def add_gaussian_noise(image, sigma=10):
    """
    Adds Gaussian noise without rescaling the original image.

    Args:
        image: The input image array (e.g., range 0-280).
        sigma: Standard deviation of the noise (intensity of the grain).
               For an image 0-255, a sigma of 5-10 is usually realistic.
    """
    # 1. Generate noise with the same shape as the image
    # loc=0 means the noise is centered at 0 (adds/subtracts equally)
    noise = np.random.normal(loc=0, scale=sigma, size=image.shape)

    # 2. Add noise to the original image
    noisy_image = image + noise

    # 3. Clip strictly to keep values valid (e.g. no negative numbers)
    # keeping the upper bound open or very high to avoid the previous error
    noisy_image = np.clip(noisy_image, 0, None)

    return noisy_image

def min_max_normalize(image, output_range=(0, 1)):
    """
    Min-Max normalization: scale to [min, max] range.

    Formula: x_norm = (x - x_min) / (x_max - x_min) * (out_max - out_min) + out_min
    """
    image_min = np.min(image)
    image_max = np.max(image)

    out_min, out_max = output_range

    normalized = (image - image_min) / (image_max - image_min)
    normalized = normalized * (out_max - out_min) + out_min

    return normalized

def z_score_normalize(image):
    """
    Z-score normalization: mean=0, std=1.

    Formula: x_norm = (x - mean(x)) / std(x)

    Better for MRI where absolute intensities are meaningless.
    """
    mean = np.mean(image)
    std = np.std(image)

    if std == 0:
        return image - mean

    return (image - mean) / std

def percentile_normalize(image, lower_percentile=1, upper_percentile=99):
    """
    Percentile-based normalization: robust to outliers.

    Clips extreme values before normalization.
    Useful when images have bright artifacts or saturated pixels.
    """
    p_low = np.percentile(image, lower_percentile)
    p_high = np.percentile(image, upper_percentile)

    # Clip to percentile range
    image_clipped = np.clip(image, p_low, p_high)

    # Min-max normalize
    normalized = (image_clipped - p_low) / (p_high - p_low)

    return normalized

# Create synthetic medical image with varying intensities
def create_synthetic_mri(size=256):
    """Synthetic MRI with bias field and different tissue intensities."""
    image = np.zeros((size, size))

    # Add tissues with different intensities
    y, x = np.ogrid[:size, :size]
    center_y, center_x = size // 2, size // 2

    # White matter (bright)
    wm_mask = ((x - center_x)**2 + (y - center_y)**2) < (size * 0.25)**2
    image[wm_mask] = 80

    # Gray matter (medium)
    gm_mask = ((x - center_x)**2 + (y - center_y)**2) < (size * 0.35)**2
    gm_mask = gm_mask & ~wm_mask
    image[gm_mask] = 10

    # CSF (dark)
    csf_mask = ((x - center_x)**2 + (y - center_y)**2) < (size * 0.15)**2
    image[csf_mask] = 1

    # Add smooth bias field (intensity inhomogeneity)
    bias_field = np.outer(
        np.linspace(0.7, 1.3, size),
        np.linspace(0.8, 1.2, size)
    )
    image = image * bias_field

    # Add noise
    image = add_gaussian_noise(image, sigma=10)

    return np.clip(image, 0, 80)

# Generate test image
mri_image = create_synthetic_mri(256)

# Apply different normalization methods
minmax_norm = min_max_normalize(mri_image)
zscore_norm = z_score_normalize(mri_image)
percentile_norm = percentile_normalize(mri_image, 1, 99)

# Visualize
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Original
axes[0, 0].imshow(mri_image, cmap='gray')
axes[0, 0].set_title('Original MRI', fontsize=12, weight='bold')
axes[0, 0].axis('off')

# Min-Max
axes[0, 1].imshow(minmax_norm, cmap='gray')
axes[0, 1].set_title('Min-Max [0,1]', fontsize=12, weight='bold')
axes[0, 1].axis('off')

# Z-score
axes[0, 2].imshow(zscore_norm, cmap='gray', vmin=-3, vmax=3)
axes[0, 2].set_title('Z-Score (mean=0, std=1)', fontsize=12, weight='bold')
axes[0, 2].axis('off')

# Percentile
axes[0, 3].imshow(percentile_norm, cmap='gray')
axes[0, 3].set_title('Percentile (1-99%)', fontsize=12, weight='bold')
axes[0, 3].axis('off')

# Histograms
axes[1, 0].hist(mri_image.ravel(), bins=50, alpha=0.7, color='blue')
axes[1, 0].set_title('Original Histogram')
axes[1, 0].set_xlabel('Intensity')
axes[1, 0].set_ylabel('Frequency')

axes[1, 1].hist(minmax_norm.ravel(), bins=50, alpha=0.7, color='green')
axes[1, 1].set_title('Min-Max Histogram')
axes[1, 1].set_xlabel('Intensity')

axes[1, 2].hist(zscore_norm.ravel(), bins=50, alpha=0.7, color='orange')
axes[1, 2].set_title('Z-Score Histogram')
axes[1, 2].set_xlabel('Intensity')

axes[1, 3].hist(percentile_norm.ravel(), bins=50, alpha=0.7, color='red')
axes[1, 3].set_title('Percentile Histogram')
axes[1, 3].set_xlabel('Intensity')

plt.suptitle('Intensity Normalization Methods for MRI', fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Normalization Statistics:")
print(f"Original     - Range: [{mri_image.min():.3f}, {mri_image.max():.3f}], Mean: {mri_image.mean():.3f}, Std: {mri_image.std():.3f}")
print(f"Min-Max      - Range: [{minmax_norm.min():.3f}, {minmax_norm.max():.3f}], Mean: {minmax_norm.mean():.3f}, Std: {minmax_norm.std():.3f}")
print(f"Z-Score      - Range: [{zscore_norm.min():.3f}, {zscore_norm.max():.3f}], Mean: {zscore_norm.mean():.3f}, Std: {zscore_norm.std():.3f}")
print(f"Percentile   - Range: [{percentile_norm.min():.3f}, {percentile_norm.max():.3f}], Mean: {percentile_norm.mean():.3f}, Std: {percentile_norm.std():.3f}")


📊 Normalization Statistics:
Original     - Range: [0.000, 80.000], Mean: 14.242, Std: 24.043
Min-Max      - Range: [0.000, 1.000], Mean: 0.178, Std: 0.301
Z-Score      - Range: [-0.592, 2.735], Mean: 0.000, Std: 1.000
Percentile   - Range: [0.000, 1.000], Mean: 0.178, Std: 0.301


## Part 2: CT Windowing (Hounsfield Units)

### What is CT Windowing?

CT scans use **Hounsfield Units (HU)** - standardized scale:
- **Air**: -1000 HU
- **Lung tissue**: -900 to -500 HU
- **Fat**: -100 to -50 HU
- **Water**: 0 HU
- **Soft tissue**: +40 to +80 HU
- **Bone**: +400 to +1000 HU
- **Metal**: +1000+ HU

**Window = Range of HU values displayed**
- **Window Level (WL)**: Center of the range
- **Window Width (WW)**: Span of the range

Displayed range = WL − WW/2 to WL + WW/2. For example, the standard **lung window WL −600 / WW 1500** shows about **−1350 to +150 HU**.

> Note: a window is written as a level and a width (WL/WW), not as a "from–to" range. "Lung window −600 / 1500" means level −600 and width 1500, not "−600 to +1500 HU".

Different windows highlight different anatomy!

In [4]:
def apply_ct_window(image_hu, window_level, window_width):
    """
    Apply CT windowing to convert HU values to display range [0, 1].

    Args:
        image_hu: CT image in Hounsfield Units
        window_level: Center of window (WL)
        window_width: Width of window (WW)

    Returns:
        Windowed image in [0, 1] range
    """
    # Calculate min/max HU for this window
    hu_min = window_level - (window_width / 2)
    hu_max = window_level + (window_width / 2)

    # Clip and normalize to [0, 1]
    windowed = np.clip(image_hu, hu_min, hu_max)
    windowed = (windowed - hu_min) / (hu_max - hu_min)

    return windowed

# Create synthetic CT scan in HU
def create_synthetic_ct_hu(size=256):
    """Synthetic chest CT with realistic HU values."""
    ct_image = np.ones((size, size)) * (-1000)  # Start with air

    y, x = np.ogrid[:size, :size]
    center_y, center_x = size // 2, size // 2

    # Body outline (soft tissue)
    body_mask = ((x - center_x)**2 + (y - center_y)**2) < (size * 0.4)**2
    ct_image[body_mask] = np.random.normal(40, 10, ct_image[body_mask].shape)  # Soft tissue

    # Lung fields (left and right)
    left_lung = ((x - center_x + 50)**2 + (y - center_y)**2) < (size * 0.2)**2
    right_lung = ((x - center_x - 50)**2 + (y - center_y)**2) < (size * 0.2)**2
    lung_mask = left_lung | right_lung
    ct_image[lung_mask] = np.random.normal(-700, 50, ct_image[lung_mask].shape)  # Lung tissue

    # Mediastinum (between lungs)
    mediastinum = (np.abs(x - center_x) < 20) & (np.abs(y - center_y) < 60)
    ct_image[mediastinum] = np.random.normal(35, 5, ct_image[mediastinum].shape)  # Soft tissue

    # Ribs (horizontal bright lines)
    for rib_y in [center_y - 60, center_y - 30, center_y, center_y + 30, center_y + 60]:
        if 0 <= rib_y < size:
            rib_mask = (np.abs(y - rib_y) < 3) & body_mask
            ct_image[rib_mask] = np.random.normal(600, 50, ct_image[rib_mask].shape)  # Bone

    # Nodule (suspicious finding in lung)
    nodule_y, nodule_x = center_y - 30, center_x + 60
    nodule_mask = ((x - nodule_x)**2 + (y - nodule_y)**2) < 20
    ct_image[nodule_mask] = np.random.normal(-200, 30, ct_image[nodule_mask].shape)  # Solid nodule

    return ct_image

# Generate synthetic CT
ct_hu = create_synthetic_ct_hu(256)

# Define standard clinical windows
windows = {
    'Lung': {'level': -600, 'width': 1500},   # ≈ -1350 to +150 HU
    'Mediastinum': {'level': 40, 'width': 400},
    'Bone': {'level': 400, 'width': 1800},
    'Brain': {'level': 40, 'width': 80},
}

# Apply windows
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

# Original (no windowing, just min-max)
axes[0].imshow(min_max_normalize(ct_hu), cmap='gray')
axes[0].set_title('Original CT\n(No windowing)', fontsize=12, weight='bold')
axes[0].axis('off')

# Apply each window
for idx, (window_name, params) in enumerate(windows.items(), start=1):
    windowed = apply_ct_window(ct_hu, params['level'], params['width'])
    axes[idx].imshow(windowed, cmap='gray')
    axes[idx].set_title(f'{window_name} Window\nWL={params["level"]}, WW={params["width"]}',
                        fontsize=11, weight='bold')
    axes[idx].axis('off')

# HU histogram
axes[5].hist(ct_hu.ravel(), bins=100, alpha=0.7, color='steelblue')
axes[5].axvline(-1000, color='red', linestyle='--', label='Air', linewidth=2)
axes[5].axvline(-500, color='orange', linestyle='--', label='Lung', linewidth=2)
axes[5].axvline(0, color='blue', linestyle='--', label='Water', linewidth=2)
axes[5].axvline(40, color='green', linestyle='--', label='Soft tissue', linewidth=2)
axes[5].axvline(600, color='purple', linestyle='--', label='Bone', linewidth=2)
axes[5].set_title('CT Histogram (Hounsfield Units)', fontsize=11, weight='bold')
axes[5].set_xlabel('HU')
axes[5].set_ylabel('Frequency')
axes[5].legend(fontsize=9)
axes[5].set_xlim(-1000, 1000)

plt.suptitle('CT Windowing for Different Anatomical Structures', fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

for window_name, params in windows.items():
    print(f"{window_name:12s} WL={params['level']:5d}, WW={params['width']:5d} → displays "
          f"{params['level'] - params['width'] / 2:+.0f} to {params['level'] + params['width'] / 2:+.0f} HU")
print("\n🔍 Windowing Effects:")
print("- Lung Window: Best for detecting nodules, emphysema, infiltrates")
print("- Mediastinum Window: Best for heart, vessels, lymph nodes")
print("- Bone Window: Best for fractures, cortical bone, calcifications")
print("- Brain Window: Best for brain tissue, gray/white matter differentiation")
print("\n🎯 Clinical Tip: Different windows reveal different pathology!")

Lung         WL= -600, WW= 1500 → displays -1350 to +150 HU
Mediastinum  WL=   40, WW=  400 → displays -160 to +240 HU
Bone         WL=  400, WW= 1800 → displays -500 to +1300 HU
Brain        WL=   40, WW=   80 → displays +0 to +80 HU

🔍 Windowing Effects:
- Lung Window: Best for detecting nodules, emphysema, infiltrates
- Mediastinum Window: Best for heart, vessels, lymph nodes
- Bone Window: Best for fractures, cortical bone, calcifications
- Brain Window: Best for brain tissue, gray/white matter differentiation

🎯 Clinical Tip: Different windows reveal different pathology!


## Part 3: MRI Bias Field Correction

### What is Bias Field?

**Bias field (intensity inhomogeneity)** is a smooth, low-frequency intensity variation in MRI:
- Caused by RF coil imperfections, patient anatomy, field strength
- Makes segmentation difficult (same tissue has different intensities!)
- Must be corrected before CNN training

**N4ITK** is the gold standard bias field correction algorithm.

In [5]:
def estimate_bias_field(image, sigma=50):
    """
    Simple bias field estimation using low-pass filtering.
    (Real implementation would use N4ITK from SimpleITK)

    Args:
        image: Input MRI
        sigma: Smoothing parameter (larger = smoother bias field)
    """
    # Smooth the image to estimate bias field
    bias_field = gaussian_filter(image, sigma=sigma)

    return bias_field

def correct_bias_field(image, bias_field, epsilon=1e-10):
    """
    Correct bias field by division.

    Corrected = Original / BiasField
    """
    # Avoid division by zero
    corrected = image / (bias_field + epsilon)

    return corrected

# Use our synthetic MRI from earlier (has bias field built in)
mri_biased = mri_image.copy()

# Estimate bias field
bias_field_estimated = estimate_bias_field(mri_biased, sigma=40)

# Correct
mri_corrected = correct_bias_field(mri_biased, bias_field_estimated)

# Normalize for visualization
mri_corrected_norm = min_max_normalize(mri_corrected)

# Visualize
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Original with bias
axes[0, 0].imshow(mri_biased, cmap='gray')
axes[0, 0].set_title('Original MRI\n(with bias field)', fontsize=12, weight='bold')
axes[0, 0].axis('off')

# Estimated bias field
axes[0, 1].imshow(bias_field_estimated, cmap='jet')
axes[0, 1].set_title('Estimated Bias Field\n(smooth intensity variation)', fontsize=12, weight='bold')
axes[0, 1].axis('off')
plt.colorbar(axes[0, 1].images[0], ax=axes[0, 1], fraction=0.046)

# Corrected
axes[0, 2].imshow(mri_corrected_norm, cmap='gray')
axes[0, 2].set_title('Bias-Corrected MRI\n(uniform intensity)', fontsize=12, weight='bold')
axes[0, 2].axis('off')

# Intensity profiles (horizontal line through center)
center_y = mri_biased.shape[0] // 2
x_coords = np.arange(mri_biased.shape[1])

axes[1, 0].plot(x_coords, mri_biased[center_y, :], label='Original', linewidth=2, color='blue')
axes[1, 0].plot(x_coords, mri_corrected_norm[center_y, :], label='Corrected', linewidth=2, color='green')
axes[1, 0].set_title('Intensity Profile (Horizontal)', fontsize=11, weight='bold')
axes[1, 0].set_xlabel('Position (pixels)')
axes[1, 0].set_ylabel('Intensity')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Histograms
axes[1, 1].hist(mri_biased.ravel(), bins=50, alpha=0.6, label='Original', color='blue')
axes[1, 1].hist(mri_corrected_norm.ravel(), bins=50, alpha=0.6, label='Corrected', color='green')
axes[1, 1].set_title('Intensity Histograms', fontsize=11, weight='bold')
axes[1, 1].set_xlabel('Intensity')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].legend()

# Show difference
difference = np.abs(min_max_normalize(mri_biased) - mri_corrected_norm)
axes[1, 2].imshow(difference, cmap='hot')
axes[1, 2].set_title('Absolute Difference\n(correction effect)', fontsize=11, weight='bold')
axes[1, 2].axis('off')
plt.colorbar(axes[1, 2].images[0], ax=axes[1, 2], fraction=0.046)

plt.suptitle('MRI Bias Field Correction (Journey 4: Brain Tumor Segmentation)',
             fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

print("\n🧠 Bias Field Correction Impact:")
print(f"Original intensity std: {mri_biased.std():.4f}")
print(f"Corrected intensity std: {mri_corrected_norm.std():.4f}")
print(f"\nBias correction makes tissue intensities more uniform → better segmentation!")
print("\n⚠️ Note: This is a simplified demo. Real MRI uses N4ITK algorithm.")


🧠 Bias Field Correction Impact:
Original intensity std: 24.0432
Corrected intensity std: 0.1463

Bias correction makes tissue intensities more uniform → better segmentation!

⚠️ Note: This is a simplified demo. Real MRI uses N4ITK algorithm.


## Part 4: CLAHE (Contrast Limited Adaptive Histogram Equalization)

### What is CLAHE?

**Problem**: Standard histogram equalization can over-amplify noise

**Solution**: CLAHE applies histogram equalization locally (in tiles) with contrast limiting:
- **Adaptive**: Different transformation for different image regions
- **Contrast limited**: Prevents noise amplification
- **Useful for**: X-rays, ultrasound, low-contrast images

**Applications**:
- Enhance subtle pathology (e.g., ground-glass opacities in COVID-19)
- Improve CNN feature extraction in low-contrast regions

In [6]:
# Create low-contrast X-ray
def create_low_contrast_xray(size=256):
    """Synthetic low-contrast chest X-ray."""
    xray = np.ones((size, size)) * 0.4  # Gray background

    y, x = np.ogrid[:size, :size]
    center_y, center_x = size // 2, size // 2

    # Lung fields (slightly darker)
    left_lung = ((x - center_x + 50)**2 + (y - center_y)**2) < (size * 0.25)**2
    right_lung = ((x - center_x - 50)**2 + (y - center_y)**2) < (size * 0.25)**2
    xray[left_lung | right_lung] = 0.35

    # Subtle opacity (pathology - hard to see!)
    opacity_y, opacity_x = center_y + 30, center_x - 60
    opacity_mask = ((x - opacity_x)**2 + (y - opacity_y)**2) < 400
    xray[opacity_mask] = 0.42  # Very subtle!

    # Add noise
    xray = random_noise(xray, mode='gaussian', var=0.005)

    return np.clip(xray, 0, 1)

xray_low_contrast = create_low_contrast_xray(256)

# Apply different enhancement methods
# 1. Standard histogram equalization
xray_histeq = exposure.equalize_hist(xray_low_contrast)

# 2. CLAHE
xray_clahe = exposure.equalize_adapthist(xray_low_contrast, clip_limit=0.03)

# 3. Adaptive contrast (different clip limit)
xray_clahe_aggressive = exposure.equalize_adapthist(xray_low_contrast, clip_limit=0.1)

# Visualize
fig = plt.figure(figsize=(16, 10))

# Images
ax1 = plt.subplot(2, 4, 1)
ax1.imshow(xray_low_contrast, cmap='gray')
ax1.set_title('Original X-Ray\n(Low contrast - opacity hard to see)', fontsize=11, weight='bold')
ax1.axis('off')
# Mark subtle opacity
circle = plt.Circle((68, 158), 20, fill=False, edgecolor='red', linewidth=2)  # opacity centre (x=128-60, y=128+30)
ax1.add_patch(circle)
ax1.text(68, 185, 'Subtle opacity', color='red', fontsize=10, ha='center', weight='bold')

ax2 = plt.subplot(2, 4, 2)
ax2.imshow(xray_histeq, cmap='gray')
ax2.set_title('Histogram Equalization\n(Global, amplifies noise)', fontsize=11, weight='bold')
ax2.axis('off')

ax3 = plt.subplot(2, 4, 3)
ax3.imshow(xray_clahe, cmap='gray')
ax3.set_title('CLAHE (clip=0.03)\n(Better contrast, less noise)', fontsize=11, weight='bold')
ax3.axis('off')
circle2 = plt.Circle((68, 158), 20, fill=False, edgecolor='green', linewidth=2)
ax3.add_patch(circle2)
ax3.text(68, 185, 'Same region', color='green', fontsize=10, ha='center', weight='bold')

ax4 = plt.subplot(2, 4, 4)
ax4.imshow(xray_clahe_aggressive, cmap='gray')
ax4.set_title('CLAHE (clip=0.1)\n(Aggressive, may over-enhance)', fontsize=11, weight='bold')
ax4.axis('off')

# Histograms
ax5 = plt.subplot(2, 4, 5)
ax5.hist(xray_low_contrast.ravel(), bins=50, alpha=0.7, color='gray')
ax5.set_title('Original Histogram\n(Narrow range)', fontsize=10, weight='bold')
ax5.set_xlabel('Intensity')
ax5.set_ylabel('Frequency')

ax6 = plt.subplot(2, 4, 6)
ax6.hist(xray_histeq.ravel(), bins=50, alpha=0.7, color='blue')
ax6.set_title('Hist. Eq. Distribution\n(Uniform)', fontsize=10, weight='bold')
ax6.set_xlabel('Intensity')

ax7 = plt.subplot(2, 4, 7)
ax7.hist(xray_clahe.ravel(), bins=50, alpha=0.7, color='green')
ax7.set_title('CLAHE Distribution\n(Spread, not uniform)', fontsize=10, weight='bold')
ax7.set_xlabel('Intensity')

ax8 = plt.subplot(2, 4, 8)
ax8.hist(xray_clahe_aggressive.ravel(), bins=50, alpha=0.7, color='orange')
ax8.set_title('CLAHE Aggressive\n(More spread)', fontsize=10, weight='bold')
ax8.set_xlabel('Intensity')

plt.suptitle('CLAHE for Enhancing Subtle Pathology in Medical Images',
             fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

# Measure it: contrast-to-noise ratio (CNR) of the opacity vs the surrounding lung
yy, xx = np.ogrid[:256, :256]
d2 = (xx - 68)**2 + (yy - 158)**2
inside, ring = d2 < 400, (d2 >= 900) & (d2 < 1600)
cnrs = {}
for name, img in [('Original', xray_low_contrast), ('Hist. Eq.', xray_histeq),
                  ('CLAHE 0.03', xray_clahe), ('CLAHE 0.1', xray_clahe_aggressive)]:
    cnr = (img[inside].mean() - img[ring].mean()) / img[ring].std()
    cnrs[name] = cnr
    print(f"Opacity CNR, {name:10s}: {cnr:.2f}")
if cnrs['CLAHE 0.03'] > 1.1 * cnrs['Original']:
    print("→ CLAHE raised the opacity's contrast-to-noise ratio.")
else:
    print(f"→ CLAHE did NOT raise the CNR here ({cnrs['Original']:.2f} → {cnrs['CLAHE 0.03']:.2f}): it stretches")
    print("  displayed contrast, which can help the eye, but it amplifies the noise as well.")
    print("  Enhancement cannot create information the noise has already hidden.")

print("\n🔍 CLAHE Applications:")
print("✓ Enhance subtle lung opacities (pneumonia, COVID-19)")
print("✓ Improve vessel visibility in angiography")
print("✓ Better visualization of mammography findings")
print("✓ Preprocessing for CNN training (improves feature detection)")
print("\n⚠️ Caution: Can amplify artifacts and noise if clip_limit too high")

Opacity CNR, Original  : 0.77
Opacity CNR, Hist. Eq. : 0.77
Opacity CNR, CLAHE 0.03: 0.71
Opacity CNR, CLAHE 0.1 : 0.71
→ CLAHE did NOT raise the CNR here (0.77 → 0.71): it stretches
  displayed contrast, which can help the eye, but it amplifies the noise as well.
  Enhancement cannot create information the noise has already hidden.

🔍 CLAHE Applications:
✓ Enhance subtle lung opacities (pneumonia, COVID-19)
✓ Improve vessel visibility in angiography
✓ Better visualization of mammography findings
✓ Preprocessing for CNN training (improves feature detection)

⚠️ Caution: Can amplify artifacts and noise if clip_limit too high


## Part 5: Image Quality Assessment

### Quality Metrics

Before feeding images to CNNs, assess quality:
1. **Signal-to-Noise Ratio (SNR)**: Higher = better
2. **Contrast-to-Noise Ratio (CNR)**: Tissue contrast relative to noise
3. **Sharpness/Blur**: Edge strength
4. **Artifact detection**: Motion, metal, aliasing

In [7]:
def calculate_snr(image, signal_roi, noise_roi):
    """
    Calculate Signal-to-Noise Ratio.

    SNR = mean(signal) / std(noise)

    Args:
        image: Input image
        signal_roi: (y1, y2, x1, x2) for signal region
        noise_roi: (y1, y2, x1, x2) for noise region (e.g., background)
    """
    signal_region = image[signal_roi[0]:signal_roi[1], signal_roi[2]:signal_roi[3]]
    noise_region = image[noise_roi[0]:noise_roi[1], noise_roi[2]:noise_roi[3]]

    signal_mean = np.mean(signal_region)
    noise_std = np.std(noise_region)

    snr = signal_mean / (noise_std + 1e-10)

    return snr

def calculate_cnr(image, roi1, roi2, noise_roi):
    """
    Calculate Contrast-to-Noise Ratio between two tissues.

    CNR = |mean(tissue1) - mean(tissue2)| / std(noise)
    """
    tissue1 = image[roi1[0]:roi1[1], roi1[2]:roi1[3]]
    tissue2 = image[roi2[0]:roi2[1], roi2[2]:roi2[3]]
    noise_region = image[noise_roi[0]:noise_roi[1], noise_roi[2]:noise_roi[3]]

    contrast = np.abs(np.mean(tissue1) - np.mean(tissue2))
    noise_std = np.std(noise_region)

    cnr = contrast / (noise_std + 1e-10)

    return cnr

def calculate_sharpness(image):
    """
    Estimate image sharpness using gradient magnitude.
    Higher = sharper edges.
    """
    # Sobel filters for gradients
    grad_x = ndimage.sobel(image, axis=1)
    grad_y = ndimage.sobel(image, axis=0)

    # Gradient magnitude
    grad_magnitude = np.sqrt(grad_x**2 + grad_y**2)

    # Average gradient strength = sharpness
    sharpness = np.mean(grad_magnitude)

    return sharpness

# Create images with different quality
high_quality = create_synthetic_mri(256)
noisy = random_noise(high_quality, mode='gaussian', var=0.05)  # Add lots of noise
blurred = gaussian_filter(high_quality, sigma=3)  # Blur

# Define ROIs for quality assessment
signal_roi = (80, 120, 80, 120)  # White matter region
noise_roi = (10, 40, 10, 40)  # Background region
tissue2_roi = (100, 140, 100, 140)  # Gray matter region

# Calculate metrics
images = {
    'High Quality': high_quality,
    'Noisy': noisy,
    'Blurred': blurred
}

results = {}
for name, img in images.items():
    snr = calculate_snr(img, signal_roi, noise_roi)
    cnr = calculate_cnr(img, signal_roi, tissue2_roi, noise_roi)
    sharpness = calculate_sharpness(img)

    results[name] = {
        'SNR': snr,
        'CNR': cnr,
        'Sharpness': sharpness
    }

# Visualize
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Images
for idx, (name, img) in enumerate(images.items()):
    axes[0, idx].imshow(img, cmap='gray')
    axes[0, idx].set_title(f'{name}\nSNR={results[name]["SNR"]:.2f}',
                           fontsize=12, weight='bold')
    axes[0, idx].axis('off')

    # Draw ROIs
    from matplotlib.patches import Rectangle
    rect_signal = Rectangle((signal_roi[2], signal_roi[0]),
                             signal_roi[3]-signal_roi[2], signal_roi[1]-signal_roi[0],
                             linewidth=2, edgecolor='red', facecolor='none')
    axes[0, idx].add_patch(rect_signal)

# Metrics comparison
metrics = ['SNR', 'CNR', 'Sharpness']
for idx, metric in enumerate(metrics):
    values = [results[name][metric] for name in images.keys()]
    axes[1, idx].bar(images.keys(), values, color=['green', 'orange', 'red'], alpha=0.7)
    axes[1, idx].set_title(f'{metric} Comparison', fontsize=12, weight='bold')
    axes[1, idx].set_ylabel(metric)
    axes[1, idx].grid(True, alpha=0.3, axis='y')

    # Annotate bars
    for i, v in enumerate(values):
        axes[1, idx].text(i, v + max(values)*0.02, f'{v:.2f}',
                          ha='center', fontsize=10, weight='bold')

plt.suptitle('Image Quality Assessment Metrics', fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Quality Assessment Results:")
print("="*60)
for name, metrics in results.items():
    print(f"\n{name}:")
    print(f"  SNR: {metrics['SNR']:.2f} (higher is better)")
    print(f"  CNR: {metrics['CNR']:.2f} (higher is better)")
    print(f"  Sharpness: {metrics['Sharpness']:.4f} (higher is better)")

print("\n" + "="*60)
print("\n🎯 Clinical Implications:")
print("- Low SNR → Poor image quality → CNN may struggle")
print("- Low CNR → Hard to distinguish tissues → Segmentation fails")
print("- Low Sharpness → Motion blur or poor resolution → Missed findings")
print("\n⚠️ Quality control is essential before CNN deployment!")


📊 Quality Assessment Results:

High Quality:
  SNR: 7.54 (higher is better)
  CNR: 6.77 (higher is better)
  Sharpness: 31.8449 (higher is better)

Noisy:
  SNR: 1.85 (higher is better)
  CNR: 0.62 (higher is better)
  Sharpness: 1.6678 (higher is better)

Blurred:
  SNR: 98.19 (higher is better)
  CNR: 87.30 (higher is better)
  Sharpness: 6.6115 (higher is better)


🎯 Clinical Implications:
- Low SNR → Poor image quality → CNN may struggle
- Low CNR → Hard to distinguish tissues → Segmentation fails
- Low Sharpness → Motion blur or poor resolution → Missed findings

⚠️ Quality control is essential before CNN deployment!


## Part 6: Complete Preprocessing Pipeline

Let's put it all together into a reusable preprocessing pipeline:

In [8]:
class MedicalImagePreprocessor:
    """
    Complete preprocessing pipeline for medical images.
    Handles CT, MRI, and X-ray modalities.
    """

    def __init__(self, modality='ct', target_size=(256, 256)):
        """
        Args:
            modality: 'ct', 'mri', or 'xray'
            target_size: Output image size
        """
        self.modality = modality.lower()
        self.target_size = target_size

        # Modality-specific parameters
        if self.modality == 'ct':
            self.window_level = -600  # Lung window by default (WL -600 / WW 1500)
            self.window_width = 1500

        print(f"✓ Preprocessor initialized for {modality.upper()} images")

    def preprocess(self, image, apply_clahe=False, correct_bias=False):
        """
        Complete preprocessing pipeline.

        Steps:
        1. Modality-specific processing
        2. Optional bias correction (MRI)
        3. Optional CLAHE (low-contrast images)
        4. Normalization
        5. Resize
        """
        processed = image.copy()

        # Step 1: Modality-specific
        if self.modality == 'ct':
            # Apply windowing
            processed = apply_ct_window(processed, self.window_level, self.window_width)

        elif self.modality == 'mri':
            # Bias field correction
            if correct_bias:
                bias_field = estimate_bias_field(processed, sigma=40)
                processed = correct_bias_field(processed, bias_field)

            # Z-score normalization (MRI has arbitrary units)
            processed = z_score_normalize(processed)
            # Clip to reasonable range for visualization
            processed = np.clip(processed, -3, 3)
            processed = min_max_normalize(processed)

        elif self.modality == 'xray':
            # CLAHE for contrast enhancement
            if apply_clahe:
                processed = exposure.equalize_adapthist(processed, clip_limit=0.03)
            else:
                processed = min_max_normalize(processed)

        # Step 2: Resize if needed
        if processed.shape != self.target_size:
            from skimage.transform import resize
            processed = resize(processed, self.target_size,
                              anti_aliasing=True, preserve_range=True)

        # Step 3: Final normalization to [0, 1]
        processed = min_max_normalize(processed)

        return processed

    def set_ct_window(self, level, width):
        """Change CT windowing parameters."""
        self.window_level = level
        self.window_width = width
        print(f"✓ CT window updated: WL={level}, WW={width}")

# Demonstrate pipeline on different modalities
print("\n" + "="*70)
print("PREPROCESSING PIPELINE DEMONSTRATION")
print("="*70)

# CT preprocessing
print("\n1. CT Preprocessing:")
ct_preprocessor = MedicalImagePreprocessor(modality='ct', target_size=(224, 224))
ct_original = create_synthetic_ct_hu(256)
ct_processed = ct_preprocessor.preprocess(ct_original)
print(f"   Input shape: {ct_original.shape} → Output shape: {ct_processed.shape}")
print(f"   Output range: [{ct_processed.min():.3f}, {ct_processed.max():.3f}]")

# MRI preprocessing
print("\n2. MRI Preprocessing:")
mri_preprocessor = MedicalImagePreprocessor(modality='mri', target_size=(224, 224))
mri_original = create_synthetic_mri(256)
mri_processed = mri_preprocessor.preprocess(mri_original, correct_bias=True)
print(f"   Input shape: {mri_original.shape} → Output shape: {mri_processed.shape}")
print(f"   Output range: [{mri_processed.min():.3f}, {mri_processed.max():.3f}]")

# X-ray preprocessing
print("\n3. X-Ray Preprocessing:")
xray_preprocessor = MedicalImagePreprocessor(modality='xray', target_size=(224, 224))
xray_original = create_low_contrast_xray(256)
xray_processed = xray_preprocessor.preprocess(xray_original, apply_clahe=True)
print(f"   Input shape: {xray_original.shape} → Output shape: {xray_processed.shape}")
print(f"   Output range: [{xray_processed.min():.3f}, {xray_processed.max():.3f}]")

# Visualize all
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(ct_original, cmap='gray', vmin=-1000, vmax=1000)
axes[0, 0].set_title('CT Original (HU)', fontsize=12, weight='bold')
axes[0, 0].axis('off')

axes[1, 0].imshow(ct_processed, cmap='gray')
axes[1, 0].set_title('CT Preprocessed\n(Windowed + Resized)', fontsize=11, weight='bold')
axes[1, 0].axis('off')

axes[0, 1].imshow(mri_original, cmap='gray')
axes[0, 1].set_title('MRI Original (Biased)', fontsize=12, weight='bold')
axes[0, 1].axis('off')

axes[1, 1].imshow(mri_processed, cmap='gray')
axes[1, 1].set_title('MRI Preprocessed\n(Bias corrected + Normalized)', fontsize=11, weight='bold')
axes[1, 1].axis('off')

axes[0, 2].imshow(xray_original, cmap='gray')
axes[0, 2].set_title('X-Ray Original (Low contrast)', fontsize=12, weight='bold')
axes[0, 2].axis('off')

axes[1, 2].imshow(xray_processed, cmap='gray')
axes[1, 2].set_title('X-Ray Preprocessed\n(CLAHE + Resized)', fontsize=11, weight='bold')
axes[1, 2].axis('off')

plt.suptitle('Complete Preprocessing Pipeline for Different Modalities',
             fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("✓ Preprocessing pipeline complete!")
print("  All images now standardized and ready for CNN training.")
print("="*70)


PREPROCESSING PIPELINE DEMONSTRATION

1. CT Preprocessing:
✓ Preprocessor initialized for CT images
   Input shape: (256, 256) → Output shape: (224, 224)
   Output range: [0.000, 1.000]

2. MRI Preprocessing:
✓ Preprocessor initialized for MRI images


   Input shape: (256, 256) → Output shape: (224, 224)
   Output range: [0.000, 1.000]

3. X-Ray Preprocessing:
✓ Preprocessor initialized for XRAY images
   Input shape: (256, 256) → Output shape: (224, 224)
   Output range: [0.000, 1.000]

✓ Preprocessing pipeline complete!
  All images now standardized and ready for CNN training.


## Part 7: Does Preprocessing Change Model Performance?

So far we have looked at preprocessed images. Now we **measure** the effect on a model.

**Experiment (synthetic MRI-like patches):**
- Task: does a 24×24 patch contain a small bright lesion?
- Training data come from "scanners" with gain 0.8–1.2 and offset ±10 (arbitrary MRI units)
- Test data: held-out patches from the same scanners, and patches from a **new scanner** with gain 1.5–2.5 and offset +50 to +150
- Classifier: logistic regression (scikit-learn) on the patch's sorted intensities (a location-free summary)
- We train it twice: on **raw** intensities, and after **per-image z-score normalisation** (Part 1)

In [9]:
from sklearn.linear_model import LogisticRegression

rng_exp = np.random.default_rng(3)

def make_scans(n, gain_range, offset_range, size=24):
    """Synthetic MRI-like patches; label 1 = small bright lesion. Each scanner applies its own gain/offset."""
    yy, xx = np.mgrid[:size, :size]
    X, y = [], rng_exp.integers(0, 2, n)
    for label in y:
        img = 100 + rng_exp.normal(0, 8, (size, size))                 # tissue
        if label:
            cy, cx = rng_exp.integers(6, size - 6, 2)
            img[(yy - cy)**2 + (xx - cx)**2 <= rng_exp.uniform(2, 4)**2] += 15   # subtle lesion
        X.append(img * rng_exp.uniform(*gain_range) + rng_exp.uniform(*offset_range))
    return np.array(X), y

def zscore_each(X):
    return (X - X.mean(axis=(1, 2), keepdims=True)) / X.std(axis=(1, 2), keepdims=True)

def intensity_features(X):
    return np.sort(X.reshape(len(X), -1), axis=1)[:, ::8]   # sorted intensities, every 8th

X_tr, y_tr = make_scans(300, (0.8, 1.2), (-10, 10))      # training scanners
X_same, y_same = make_scans(300, (0.8, 1.2), (-10, 10))  # held out, same scanners
X_new, y_new = make_scans(300, (1.5, 2.5), (50, 150))    # new scanner

norm_results = {}
for name, prep in [('Raw intensities', lambda X: X), ('Z-score per image', zscore_each)]:
    clf = LogisticRegression(max_iter=5000).fit(intensity_features(prep(X_tr)), y_tr)
    norm_results[name] = (clf.score(intensity_features(prep(X_same)), y_same),
                          clf.score(intensity_features(prep(X_new)), y_new))

fig, ax = plt.subplots(figsize=(8, 4.5))
xpos = np.arange(2)
for j, (name, (acc_same, acc_new)) in enumerate(norm_results.items()):
    bars = ax.bar(xpos + (j - 0.5) * 0.35, [acc_same, acc_new], 0.35, label=name)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f'{bar.get_height():.2f}', ha='center')
ax.set_xticks(xpos)
ax.set_xticklabels(['Same scanners (held out)', 'New scanner'])
ax.set_ylim(0.4, 1.0)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1)
ax.set_ylabel('Accuracy')
ax.set_title('Lesion classifier with vs without intensity normalisation', fontsize=12, weight='bold')
ax.legend()
plt.tight_layout()
plt.show()

for name, (acc_same, acc_new) in norm_results.items():
    print(f"{name:18s}: same scanners {acc_same:.3f} → new scanner {acc_new:.3f} (drop {acc_same - acc_new:+.3f})")
raw_drop = norm_results['Raw intensities'][0] - norm_results['Raw intensities'][1]
z_drop = norm_results['Z-score per image'][0] - norm_results['Z-score per image'][1]
if raw_drop > z_drop + 0.05:
    z_text = 'no loss' if z_drop <= 0 else f'only {z_drop:.0%}'
    print(f"\n→ Without normalisation the model loses {raw_drop:.0%} accuracy on the new scanner; with z-score {z_text}.")
    print("  The raw model learned the training scanners' intensity scale, not the lesion.")
else:
    print("\n→ In this run normalisation did not clearly protect against the scanner shift.")

Raw intensities   : same scanners 0.827 → new scanner 0.647 (drop +0.180)
Z-score per image : same scanners 0.820 → new scanner 0.837 (drop -0.017)

→ Without normalisation the model loses 18% accuracy on the new scanner; with z-score no loss.
  The raw model learned the training scanners' intensity scale, not the lesion.


### Colour Constancy for Dermoscopy (Journey 5: Priya)

Dermoscopy and smartphone photos are taken under different lights, so the same skin can look warmer, cooler or greener. **Shades of Grey** colour constancy (Finlayson & Trezzi, 2004; widely used in ISIC skin-lesion challenges) estimates the colour of the light from the image itself and divides it out:

$$e_c = \Big(\frac{1}{N}\sum_{x} I_c(x)^p\Big)^{1/p}, \qquad I'_c(x) = I_c(x) \cdot \frac{\bar e}{e_c}$$

with $p = 6$ (p = 1 is "grey world", p = ∞ is "max-RGB"). Below, one synthetic lesion photographed under three colour casts is corrected, and we measure how far apart the three versions are before and after.

In [10]:
def shades_of_grey(img, p=6):
    """Shades-of-Grey colour constancy for an RGB image in [0, 1]."""
    illum = np.power(np.mean(np.power(img, p), axis=(0, 1)), 1 / p)   # estimated light colour per channel
    return np.clip(img * (illum.mean() / illum), 0, 1)

# Synthetic dermoscopy-like image: skin + brown lesion (Fitzpatrick IV-like tone)
size = 96
yy, xx = np.mgrid[:size, :size]
derm = np.ones((size, size, 3)) * np.array([0.76, 0.57, 0.41]) + np.random.normal(0, 0.02, (size, size, 3))
lesion = (yy - 48)**2 / 20**2 + (xx - 46)**2 / 15**2 <= 1
derm[lesion] = np.array([0.35, 0.22, 0.14]) + np.random.normal(0, 0.03, (lesion.sum(), 3))
derm = np.clip(derm, 0, 1)

casts = {'Warm light': np.array([1.15, 0.95, 0.75]),
         'Cool light': np.array([0.80, 0.95, 1.20]),
         'Greenish light': np.array([0.90, 1.15, 0.85])}
photos = {name: np.clip(derm * c, 0, 1) for name, c in casts.items()}
corrected = {name: shades_of_grey(img) for name, img in photos.items()}

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for j, name in enumerate(casts):
    axes[0, j].imshow(photos[name]); axes[0, j].set_title(f'{name}\n(as captured)', fontsize=11, weight='bold'); axes[0, j].axis('off')
    axes[1, j].imshow(corrected[name]); axes[1, j].set_title('After Shades of Grey (p=6)', fontsize=11, weight='bold'); axes[1, j].axis('off')
plt.suptitle('Colour constancy for dermoscopy (synthetic lesion, three lighting conditions)', fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

def lesion_chromaticity(img):
    rgb = img[lesion].mean(axis=0)
    return rgb / rgb.sum()                  # colour independent of brightness

def spread(images):
    chroma = np.array([lesion_chromaticity(im) for im in images.values()])
    return np.abs(chroma[:, None, :] - chroma[None, :, :]).sum(axis=2).max()   # largest pairwise difference

spread_before, spread_after = spread(photos), spread(corrected)
print(f"Lesion colour spread across the 3 lights: before {spread_before:.3f}, after {spread_after:.3f} "
      f"(max pairwise L1 distance of r,g,b chromaticity)")
if spread_after < 0.5 * spread_before:
    print(f"→ Shades of Grey removed {1 - spread_after / spread_before:.0%} of the lighting-induced colour difference,")
    print("  so a model sees the lesion's colour, not the lamp's. Colour is diagnostic in melanoma (ABCDE).")
    print("  (The synthetic casts here are pure per-channel gains, the ideal case; real lighting is messier.)")
else:
    print("→ The correction did not remove most of the colour difference in this example.")
print("⚠️ Colour constancy assumes the scene averages to grey; on a large, very dark lesion it can over-correct.")

Lesion colour spread across the 3 lights: before 0.271, after 0.000 (max pairwise L1 distance of r,g,b chromaticity)
→ Shades of Grey removed 100% of the lighting-induced colour difference,
  so a model sees the lesion's colour, not the lamp's. Colour is diagnostic in melanoma (ABCDE).
  (The synthetic casts here are pure per-channel gains, the ideal case; real lighting is messier.)
⚠️ Colour constancy assumes the scene averages to grey; on a large, very dark lesion it can over-correct.


## Summary: Key Takeaways

### What We Learned

1. **Intensity Normalization**
   - **Min-Max**: Scale to fixed range [0, 1] or [-1, 1]
   - **Z-Score**: Standardize to mean=0, std=1 (best for MRI)
   - **Percentile**: Robust to outliers and artifacts
   - Critical for CNN training across different scanners

2. **CT Windowing**
   - Hounsfield Units (HU) are standardized but span huge range
   - Different windows highlight different anatomy:
     - Lung window (WL −600, WW 1500 ≈ −1350 to +150 HU): Nodules, infiltrates
     - Mediastinum window (40 HU, 400 width): Heart, vessels
     - Bone window (400 HU, 1800 width): Fractures, calcifications
   - Essential for Journey 3 (Lung Nodule Detection)

3. **MRI Bias Field Correction**
   - Intensity inhomogeneity from RF coils, anatomy
   - Makes segmentation difficult (same tissue ≠ same intensity)
   - N4ITK algorithm is gold standard
   - Critical for Journey 4 (Brain Tumor Segmentation)

4. **CLAHE (Contrast Enhancement)**
   - Adaptive histogram equalization with contrast limiting
   - Stretches displayed contrast, which can help the human eye on low-contrast images (X-rays, ultrasound)
   - But it amplifies noise too: in Part 4 it did not raise the opacity's contrast-to-noise ratio (CNR). Enhancement cannot create information the noise has already hidden
   - Whether it helps a CNN must be measured, not assumed

5. **Measured effect on a model** (Part 7)
   - Without intensity normalisation, a classifier learns the training scanners' intensity scale and degrades on a new scanner
   - Shades-of-Grey colour constancy removes lighting colour casts from dermoscopy images

6. **Image Quality Assessment**
   - **SNR**: Signal-to-Noise Ratio (higher = better)
   - **CNR**: Contrast-to-Noise Ratio (tissue differentiation)
   - **Sharpness**: Edge strength (motion blur detection)
   - Read them together: in Part 5 the blurred image has the HIGHEST SNR and CNR (blurring averages the noise away) but low sharpness, so no single metric is enough
   - Quality control prevents "garbage in, garbage out"

### Connections to Clinical Journeys

- **Journey 3 (Jamal)**: CT windowing for lung nodule visualization
- **Journey 4 (Elena)**: MRI bias correction for tumor segmentation
- **Journey 5 (Priya)**: Colour constancy (Shades of Grey) and illumination correction (dermoscopy)

### Real-World Considerations

**Preprocessing Best Practices:**
- Always inspect images before AND after preprocessing
- Use modality-appropriate techniques (don't apply CT windowing to MRI!)
- Document preprocessing parameters (reproducibility)
- Validate that CNN performs well on preprocessed data
- Consider domain-specific requirements (e.g., radiologist windowing preferences)

**Common Pitfalls:**
- Over-aggressive CLAHE → amplified noise, artifacts
- Wrong CT window → missed pathology
- Ignoring bias field → poor MRI segmentation
- No quality control → "garbage in, garbage out"
- Inconsistent preprocessing → poor CNN generalization

**Regulatory Considerations:**
- FDA 510(k) requires detailed preprocessing documentation
- Changes to preprocessing = new regulatory submission
- Must validate on diverse scanners/protocols

---

## Exercises

1. **Which normalisation survives a new scanner?** (Part 7): add `min_max_normalize` and `percentile_normalize` (Part 1) as extra entries next to `zscore_each` in the loop that fills `norm_results` (apply them per image). Which methods keep accuracy on `X_new`? Then add a few bright outlier pixels (metal artefact) to some scans: which method breaks?

2. **CLAHE and noise** (Part 4): vary `clip_limit` from 0.01 to 0.2 and the noise level in `create_low_contrast_xray`, and recompute the opacity CNR with the `inside`/`ring` masks. Is there any setting where CLAHE raises the CNR rather than only the displayed contrast?

3. **A quality-control rule that catches blur** (Part 5): the `blurred` image scores the highest SNR and CNR. Write a rule using `calculate_snr` and `calculate_sharpness` that flags both `noisy` and `blurred` but passes `high_quality`. Test it on blur sigmas 1-4 and noise variances 0.005-0.05: where does your rule start flagging?

4. **Bias field severity** (Parts 1 and 3): in `create_synthetic_mri`, change the `bias_field` ramps (`np.linspace(0.7, 1.3, size)`) to weaker (0.95-1.05) and stronger (0.4-1.6) versions. For each, compare the mean intensity of the same tissue (e.g. left vs right white matter) before and after `correct_bias_field`. When does correction matter, and when does `estimate_bias_field` with `sigma=40` start removing real anatomy?

5. **Windowing for a subtle finding** (Part 2): build a small synthetic lung patch at −850 HU with a ground-glass opacity at −600 HU plus noise (σ ≈ 30 HU). Apply `apply_ct_window` with the lung and mediastinal windows and compute the opacity's contrast in each. Which window makes it visible, and why?

6. **When colour constancy fails** (Part 7, dermoscopy): vary `p` in `shades_of_grey` (p=1 is grey-world, large p approaches max-RGB) and enlarge the dark `lesion` until it covers half of `derm`. How do `spread_before`/`spread_after` and the lesion's corrected colour change? This is the over-correction the warning mentions.

---

*This notebook is part of "AI in Healthcare" (Volume 1: Machine Learning Foundations)*  
*Full implementation and companion code available in the book repository*

**Next Steps:**
- **Notebook 6.4**: Data Augmentation (rotations, flips, elastic deformations)
- **Notebook 6.7**: GradCAM Interpretability (visualizing what CNNs learn)
- **Journeys 3-5**: Complete clinical implementations with full preprocessing pipelines